In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("examples zero to hero") \
    .master("local[*]") \
    .getOrCreate()

spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/23 11:44:16 WARN Utils: Your hostname, codespaces-b99eb4, resolves to a loopback address: 127.0.0.1; using 10.0.1.96 instead (on interface eth0)
25/11/23 11:44:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/23 11:44:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SECTION 1 — BASIC DATAFRAME OPERATIONS

Creating sample datframe to practice!

In [6]:
from pyspark.sql.functions import col

data = [
    ("IT",      2023, 120),
    ("IT",      2024, 150),
    ("HR",      2023, 80),
    ("HR",      2024, 90),
    ("Finance", 2023, 200),
    ("Finance", 2024, 220)
]

columns = ["dept", "year", "revenue"]

df = spark.createDataFrame(data, schema=columns)
df.show()


+-------+----+-------+
|   dept|year|revenue|
+-------+----+-------+
|     IT|2023|    120|
|     IT|2024|    150|
|     HR|2023|     80|
|     HR|2024|     90|
|Finance|2023|    200|
|Finance|2024|    220|
+-------+----+-------+



show only 2

In [7]:
df.show(2)

+----+----+-------+
|dept|year|revenue|
+----+----+-------+
|  IT|2023|    120|
|  IT|2024|    150|
+----+----+-------+
only showing top 2 rows


check datatypes and how schema look like

In [6]:
df.printSchema()

root
 |-- dept: string (nullable = true)
 |-- year: long (nullable = true)
 |-- revenue: long (nullable = true)



Get number of rows

In [7]:
df.count()

6

Get columns

In [8]:
df.columns

['dept', 'year', 'revenue']

selecting columns

In [10]:
df.select(col("dept")).show()

+-------+
|   dept|
+-------+
|     IT|
|     IT|
|     HR|
|     HR|
|Finance|
|Finance|
+-------+



In [11]:
df.select("dept","revenue").show()

+-------+-------+
|   dept|revenue|
+-------+-------+
|     IT|    120|
|     IT|    150|
|     HR|     80|
|     HR|     90|
|Finance|    200|
|Finance|    220|
+-------+-------+



Renaming column

In [13]:
df.withColumnRenamed("dept","department").show()

+----------+----+-------+
|department|year|revenue|
+----------+----+-------+
|        IT|2023|    120|
|        IT|2024|    150|
|        HR|2023|     80|
|        HR|2024|     90|
|   Finance|2023|    200|
|   Finance|2024|    220|
+----------+----+-------+



Filter rows

In [18]:
df1 = df.filter(col("revenue")>250)
df1.show()

+----+----+-------+
|dept|year|revenue|
+----+----+-------+
+----+----+-------+



filtering with AND condition

In [30]:
df1=df.filter(
    (col("dept") == "IT") &
    (col("revenue") < 100)
).show()

+----+----+-------+
|dept|year|revenue|
+----+----+-------+
+----+----+-------+



filtering with OR condition

In [8]:
df.filter((col("year")==2023) | (col("revenue")>100)).show()


+-------+----+-------+
|   dept|year|revenue|
+-------+----+-------+
|     IT|2023|    120|
|     IT|2024|    150|
|     HR|2023|     80|
|Finance|2023|    200|
|Finance|2024|    220|
+-------+----+-------+



IN condition

In [31]:
df.filter(col("dept").isin("IT")).show()

+----+----+-------+
|dept|year|revenue|
+----+----+-------+
|  IT|2023|    120|
|  IT|2024|    150|
+----+----+-------+



NOT condition (~)

In [32]:
df.filter(~col("dept").isin("IT")).show()

+-------+----+-------+
|   dept|year|revenue|
+-------+----+-------+
|     HR|2023|     80|
|     HR|2024|     90|
|Finance|2023|    200|
|Finance|2024|    220|
+-------+----+-------+



Distinct values

In [11]:
df.select("dept").distinct().show()

+-------+
|   dept|
+-------+
|     IT|
|     HR|
|Finance|
+-------+



Drop a column

In [12]:
df.drop("year").show()


+-------+-------+
|   dept|revenue|
+-------+-------+
|     IT|    120|
|     IT|    150|
|     HR|     80|
|     HR|     90|
|Finance|    200|
|Finance|    220|
+-------+-------+



Add a new column

In [15]:
df.withColumn("rev_k", col("revenue")/1000).show()


+-------+----+-------+-----+
|   dept|year|revenue|rev_k|
+-------+----+-------+-----+
|     IT|2023|    120| 0.12|
|     IT|2024|    150| 0.15|
|     HR|2023|     80| 0.08|
|     HR|2024|     90| 0.09|
|Finance|2023|    200|  0.2|
|Finance|2024|    220| 0.22|
+-------+----+-------+-----+



Cast column type

In [17]:
df.withColumn("revenue", col("revenue").cast("double")).show()


+-------+----+-------+
|   dept|year|revenue|
+-------+----+-------+
|     IT|2023|  120.0|
|     IT|2024|  150.0|
|     HR|2023|   80.0|
|     HR|2024|   90.0|
|Finance|2023|  200.0|
|Finance|2024|  220.0|
+-------+----+-------+



Replace nulls

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

spark = SparkSession.builder.master("local[*]").appName("null-practice").getOrCreate()

data = [
    (1, "Somesh",   "IT",       90000,   "2025-01-05"),
    (2, None,       "IT",       120000,  "2025-01-15"),   # name NULL
    (3, "Rohan",    None,       70000,   "2025-02-10"),   # dept NULL
    (4, "Nikhil",   "Finance",  None,    "2025-02-25"),   # salary NULL
    (5, "Priya",    "IT",       95000,   None),           # date NULL
    (6, None,       None,       None,    None),           # everything NULL
    (7, "John",     "Finance",  105000,  "2025-03-20"),
    (8, "Arjun",    "IT",       98000,   "2025-03-25"),
    (9, None,       "IT",       65000,   "2025-03-30"),   # name NULL
    (10, "Sandeep", "HR",       85000,   None)            # date NULL
]

columns = ["emp_id", "name", "dept", "salary", "join_date"]

df_nulls = spark.createDataFrame(data, columns)
df_nulls.show(truncate=False)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/23 08:22:17 WARN Utils: Your hostname, codespaces-b99eb4, resolves to a loopback address: 127.0.0.1; using 10.0.3.238 instead (on interface eth0)
25/11/23 08:22:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/23 08:22:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------+-------+-------+------+----------+
|emp_id|name   |dept   |salary|join_date |
+------+-------+-------+------+----------+
|1     |Somesh |IT     |90000 |2025-01-05|
|2     |NULL   |IT     |120000|2025-01-15|
|3     |Rohan  |NULL   |70000 |2025-02-10|
|4     |Nikhil |Finance|NULL  |2025-02-25|
|5     |Priya  |IT     |95000 |NULL      |
|6     |NULL   |NULL   |NULL  |NULL      |
|7     |John   |Finance|105000|2025-03-20|
|8     |Arjun  |IT     |98000 |2025-03-25|
|9     |NULL   |IT     |65000 |2025-03-30|
|10    |Sandeep|HR     |85000 |NULL      |
+------+-------+-------+------+----------+



In [22]:
df_nulls.fillna({"dept":"N/A"}).show()


+------+-------+-------+------+----------+
|emp_id|   name|   dept|salary| join_date|
+------+-------+-------+------+----------+
|     1| Somesh|     IT| 90000|2025-01-05|
|     2|   NULL|     IT|120000|2025-01-15|
|     3|  Rohan|    N/A| 70000|2025-02-10|
|     4| Nikhil|Finance|  NULL|2025-02-25|
|     5|  Priya|     IT| 95000|      NULL|
|     6|   NULL|    N/A|  NULL|      NULL|
|     7|   John|Finance|105000|2025-03-20|
|     8|  Arjun|     IT| 98000|2025-03-25|
|     9|   NULL|     IT| 65000|2025-03-30|
|    10|Sandeep|     HR| 85000|      NULL|
+------+-------+-------+------+----------+



In [23]:
df_nulls.dropna().show()


+------+------+-------+------+----------+
|emp_id|  name|   dept|salary| join_date|
+------+------+-------+------+----------+
|     1|Somesh|     IT| 90000|2025-01-05|
|     7|  John|Finance|105000|2025-03-20|
|     8| Arjun|     IT| 98000|2025-03-25|
+------+------+-------+------+----------+



Order by revenue desc

In [32]:
df.orderBy(col("salary").desc()).show()


+------+-------+-------+------+----------+
|emp_id|   name|   dept|salary| join_date|
+------+-------+-------+------+----------+
|     2|   NULL|     IT|120000|2025-01-15|
|     7|   John|Finance|105000|2025-03-20|
|     8|  Arjun|     IT| 98000|2025-03-25|
|     5|  Priya|     IT| 95000|      NULL|
|     1| Somesh|     IT| 90000|2025-01-05|
|    10|Sandeep|     HR| 85000|      NULL|
|     3|  Rohan|   NULL| 70000|2025-02-10|
|     9|   NULL|     IT| 65000|2025-03-30|
|     4| Nikhil|Finance|  NULL|2025-02-25|
|     6|   NULL|   NULL|  NULL|      NULL|
+------+-------+-------+------+----------+



SECTION 2 — GROUPBY & AGGREGATIONS

Count rows per dept

In [7]:
df.groupBy("dept").count().show()


+-------+-----+
|   dept|count|
+-------+-----+
|     IT|    2|
|     HR|    2|
|Finance|    2|
+-------+-----+



Sum revenue per dept

In [9]:
from pyspark.sql.functions import sum
df.groupBy("dept").agg(sum("revenue").alias("total_rev")).show()


+-------+---------+
|   dept|total_rev|
+-------+---------+
|     IT|      270|
|     HR|      170|
|Finance|      420|
+-------+---------+



In [11]:
from pyspark.sql.functions import *

Max revenue

In [12]:
df.agg(max("revenue")).show()


+------------+
|max(revenue)|
+------------+
|         220|
+------------+



Min revenue

In [13]:
df.agg(min("revenue")).show()


+------------+
|min(revenue)|
+------------+
|          80|
+------------+



Multi-column groupBy

In [14]:
df.groupBy("dept","year").agg(sum("revenue")).show()


+-------+----+------------+
|   dept|year|sum(revenue)|
+-------+----+------------+
|     IT|2023|         120|
|     IT|2024|         150|
|     HR|2023|          80|
|     HR|2024|          90|
|Finance|2023|         200|
|Finance|2024|         220|
+-------+----+------------+



Count distinct years

In [16]:
from pyspark.sql.functions import countDistinct

df.select(countDistinct("year")).show()


+--------------------+
|count(DISTINCT year)|
+--------------------+
|                   2|
+--------------------+



Group and sort

In [17]:
df.groupBy("dept").agg(sum("revenue").alias("t")).orderBy(col("t").desc()).show()


+-------+---+
|   dept|  t|
+-------+---+
|Finance|420|
|     IT|270|
|     HR|170|
+-------+---+



Add constant column

In [18]:
df.withColumn("country", lit("India")).show()


+-------+----+-------+-------+
|   dept|year|revenue|country|
+-------+----+-------+-------+
|     IT|2023|    120|  India|
|     IT|2024|    150|  India|
|     HR|2023|     80|  India|
|     HR|2024|     90|  India|
|Finance|2023|    200|  India|
|Finance|2024|    220|  India|
+-------+----+-------+-------+



Bucket revenue into categories (when, otherwise)

In [19]:
df.withColumn("level", when(col("revenue")>150,"High").otherwise("Low")).show()


+-------+----+-------+-----+
|   dept|year|revenue|level|
+-------+----+-------+-----+
|     IT|2023|    120|  Low|
|     IT|2024|    150|  Low|
|     HR|2023|     80|  Low|
|     HR|2024|     90|  Low|
|Finance|2023|    200| High|
|Finance|2024|    220| High|
+-------+----+-------+-----+



Check duplicate rows

In [ ]:
df.groupBy(df.columns).count().filter(col("count")>1).show()


Collect rows into a list

In [20]:
df.groupBy("dept").agg(collect_list("revenue")).show()


+-------+---------------------+
|   dept|collect_list(revenue)|
+-------+---------------------+
|     IT|           [120, 150]|
|     HR|             [80, 90]|
|Finance|           [200, 220]|
+-------+---------------------+



Pivot

In [21]:
df.groupBy("dept").pivot("year").agg(sum("revenue")).show()


+-------+----+----+
|   dept|2023|2024|
+-------+----+----+
|     HR|  80|  90|
|Finance| 200| 220|
|     IT| 120| 150|
+-------+----+----+



SECTION 3 — JOINS

In [24]:
data = [
    ("IT",      "Ramesh",  500000),
    ("HR",      "Sita",    200000),
    ("Finance", "Karan",   800000)
]
columns = ["dept", "manager", "budget"]
df_dept = spark.createDataFrame(data, columns)


In [25]:
df_dept.show()

+-------+-------+------+
|   dept|manager|budget|
+-------+-------+------+
|     IT| Ramesh|500000|
|     HR|   Sita|200000|
|Finance|  Karan|800000|
+-------+-------+------+



Inner join

In [28]:
df.show()

+-------+----+-------+
|   dept|year|revenue|
+-------+----+-------+
|     IT|2023|    120|
|     IT|2024|    150|
|     HR|2023|     80|
|     HR|2024|     90|
|Finance|2023|    200|
|Finance|2024|    220|
+-------+----+-------+



In [29]:
df_dept.show()

+-------+-------+------+
|   dept|manager|budget|
+-------+-------+------+
|     IT| Ramesh|500000|
|     HR|   Sita|200000|
|Finance|  Karan|800000|
+-------+-------+------+



In [26]:
df.join(df_dept,"dept","inner").show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|Finance|2023|    200|  Karan|800000|
|Finance|2024|    220|  Karan|800000|
|     HR|2023|     80|   Sita|200000|
|     HR|2024|     90|   Sita|200000|
|     IT|2023|    120| Ramesh|500000|
|     IT|2024|    150| Ramesh|500000|
+-------+----+-------+-------+------+



Left Join

In [27]:
df.join(df_dept,"dept","left").show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|     IT|2023|    120| Ramesh|500000|
|     HR|2023|     80|   Sita|200000|
|     IT|2024|    150| Ramesh|500000|
|     HR|2024|     90|   Sita|200000|
|Finance|2023|    200|  Karan|800000|
|Finance|2024|    220|  Karan|800000|
+-------+----+-------+-------+------+



Right join

In [30]:
df.join(df_dept,"dept","right").show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|     IT|2024|    150| Ramesh|500000|
|     IT|2023|    120| Ramesh|500000|
|     HR|2024|     90|   Sita|200000|
|     HR|2023|     80|   Sita|200000|
|Finance|2024|    220|  Karan|800000|
|Finance|2023|    200|  Karan|800000|
+-------+----+-------+-------+------+



Full join

In [31]:
df.join(df_dept,"dept","outer").show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|Finance|2023|    200|  Karan|800000|
|Finance|2024|    220|  Karan|800000|
|     HR|2023|     80|   Sita|200000|
|     HR|2024|     90|   Sita|200000|
|     IT|2023|    120| Ramesh|500000|
|     IT|2024|    150| Ramesh|500000|
+-------+----+-------+-------+------+



Anti join - left_anti returns only those rows from the left DataFrame that have NO MATCH in the right DataFrame.

In [32]:
df.join(df_dept,"dept","left_anti").show()


+----+----+-------+
|dept|year|revenue|
+----+----+-------+
+----+----+-------+



Semi join - left_semi = keep rows from left where a match exists in right

In [ ]:
df.join(df_dept,"dept","left_semi").show()


Broadcast join (optimize)

In [33]:
from pyspark.sql.functions import broadcast
df.join(broadcast(df_dept),"dept").show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|     IT|2023|    120| Ramesh|500000|
|     IT|2024|    150| Ramesh|500000|
|     HR|2023|     80|   Sita|200000|
|     HR|2024|     90|   Sita|200000|
|Finance|2023|    200|  Karan|800000|
|Finance|2024|    220|  Karan|800000|
+-------+----+-------+-------+------+



Self join

In [34]:
df.alias("a").join(df.alias("b"), col("a.dept")==col("b.dept")).show()


+-------+----+-------+-------+----+-------+
|   dept|year|revenue|   dept|year|revenue|
+-------+----+-------+-------+----+-------+
|Finance|2023|    200|Finance|2023|    200|
|Finance|2023|    200|Finance|2024|    220|
|Finance|2024|    220|Finance|2023|    200|
|Finance|2024|    220|Finance|2024|    220|
|     HR|2023|     80|     HR|2023|     80|
|     HR|2023|     80|     HR|2024|     90|
|     HR|2024|     90|     HR|2023|     80|
|     HR|2024|     90|     HR|2024|     90|
|     IT|2023|    120|     IT|2023|    120|
|     IT|2023|    120|     IT|2024|    150|
|     IT|2024|    150|     IT|2023|    120|
|     IT|2024|    150|     IT|2024|    150|
+-------+----+-------+-------+----+-------+



Check rows missing in dept table

In [36]:
df.join(df_dept,"dept","left_anti").show()


+----+----+-------+
|dept|year|revenue|
+----+----+-------+
+----+----+-------+



Add default budget for missing dept

In [37]:
df.join(df_dept,"dept","left").fillna({"budget":0}).show()


+-------+----+-------+-------+------+
|   dept|year|revenue|manager|budget|
+-------+----+-------+-------+------+
|     IT|2023|    120| Ramesh|500000|
|     HR|2023|     80|   Sita|200000|
|     IT|2024|    150| Ramesh|500000|
|     HR|2024|     90|   Sita|200000|
|Finance|2023|    200|  Karan|800000|
|Finance|2024|    220|  Karan|800000|
+-------+----+-------+-------+------+



Multiple conditions

In [38]:
df.join(df_dept, (df.dept==df_dept.dept) & (df.year==2023)).show()


+-------+----+-------+-------+-------+------+
|   dept|year|revenue|   dept|manager|budget|
+-------+----+-------+-------+-------+------+
|Finance|2023|    200|Finance|  Karan|800000|
|     HR|2023|     80|     HR|   Sita|200000|
|     IT|2023|    120|     IT| Ramesh|500000|
+-------+----+-------+-------+-------+------+



SECTION 4 — WINDOW FUNCTIONS

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.master("local[*]").appName("window-practice").getOrCreate()

data = [
    ("IT",       "Somesh",  2023, 120),
    ("IT",       "Ramesh",  2024, 150),
    ("IT",       "Kiran",   2024, 180),

    ("HR",       "Sita",    2023,  80),
    ("HR",       "Anita",   2024,  90),
    ("HR",       "Rekha",   2024, 110),

    ("Finance",  "Karan",   2023, 200),
    ("Finance",  "John",    2024, 220),
    ("Finance",  "Amit",    2024, 250)
]

columns = ["dept", "employee", "year", "revenue"]

df = spark.createDataFrame(data, columns)
df.show()


25/11/23 11:50:39 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-------+--------+----+-------+
|   dept|employee|year|revenue|
+-------+--------+----+-------+
|     IT|  Somesh|2023|    120|
|     IT|  Ramesh|2024|    150|
|     IT|   Kiran|2024|    180|
|     HR|    Sita|2023|     80|
|     HR|   Anita|2024|     90|
|     HR|   Rekha|2024|    110|
|Finance|   Karan|2023|    200|
|Finance|    John|2024|    220|
|Finance|    Amit|2024|    250|
+-------+--------+----+-------+



Dense rank by revenue

dense_rank()
➜ Gives same rank to ties
➜ NO skipping of next number

In [7]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
w = Window.orderBy(col("revenue").desc())
df.withColumn("rank", dense_rank().over(w)).show()


25/11/23 11:51:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+----+
|   dept|employee|year|revenue|rank|
+-------+--------+----+-------+----+
|Finance|    Amit|2024|    250|   1|
|Finance|    John|2024|    220|   2|
|Finance|   Karan|2023|    200|   3|
|     IT|   Kiran|2024|    180|   4|
|     IT|  Ramesh|2024|    150|   5|
|     IT|  Somesh|2023|    120|   6|
|     HR|   Rekha|2024|    110|   7|
|     HR|   Anita|2024|     90|   8|
|     HR|    Sita|2023|     80|   9|
+-------+--------+----+-------+----+



row_number()
➜ Gives unique number to each row.
➜ NO ties allowed (even if values are same).

In [8]:
df.withColumn("rn", row_number().over(w)).show()


25/11/23 11:51:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+---+
|   dept|employee|year|revenue| rn|
+-------+--------+----+-------+---+
|Finance|    Amit|2024|    250|  1|
|Finance|    John|2024|    220|  2|
|Finance|   Karan|2023|    200|  3|
|     IT|   Kiran|2024|    180|  4|
|     IT|  Ramesh|2024|    150|  5|
|     IT|  Somesh|2023|    120|  6|
|     HR|   Rekha|2024|    110|  7|
|     HR|   Anita|2024|     90|  8|
|     HR|    Sita|2023|     80|  9|
+-------+--------+----+-------+---+



rank()
➜ Gives same rank to ties
➜ BUT skips the next number after a tie

In [9]:
df.withColumn("r", rank().over(w)).show()


25/11/23 11:51:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:51:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+---+
|   dept|employee|year|revenue|  r|
+-------+--------+----+-------+---+
|Finance|    Amit|2024|    250|  1|
|Finance|    John|2024|    220|  2|
|Finance|   Karan|2023|    200|  3|
|     IT|   Kiran|2024|    180|  4|
|     IT|  Ramesh|2024|    150|  5|
|     IT|  Somesh|2023|    120|  6|
|     HR|   Rekha|2024|    110|  7|
|     HR|   Anita|2024|     90|  8|
|     HR|    Sita|2023|     80|  9|
+-------+--------+----+-------+---+



Top 2 revenues

In [11]:
df.withColumn("rank", dense_rank().over(w)).filter(col("rank")<=2).show()


25/11/23 11:56:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:56:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:56:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+----+
|   dept|employee|year|revenue|rank|
+-------+--------+----+-------+----+
|Finance|    Amit|2024|    250|   1|
|Finance|    John|2024|    220|   2|
+-------+--------+----+-------+----+



25/11/23 11:56:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 11:56:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Partition by dept

In [12]:
w2 = Window.partitionBy("dept").orderBy(col("revenue"))
df.withColumn("rn", row_number().over(w2)).show()


+-------+--------+----+-------+---+
|   dept|employee|year|revenue| rn|
+-------+--------+----+-------+---+
|Finance|   Karan|2023|    200|  1|
|Finance|    John|2024|    220|  2|
|Finance|    Amit|2024|    250|  3|
|     HR|    Sita|2023|     80|  1|
|     HR|   Anita|2024|     90|  2|
|     HR|   Rekha|2024|    110|  3|
|     IT|  Somesh|2023|    120|  1|
|     IT|  Ramesh|2024|    150|  2|
|     IT|   Kiran|2024|    180|  3|
+-------+--------+----+-------+---+



Top 1 per dept

In [14]:
df.withColumn("rn", row_number().over(w2)).filter(col("rn")==1).show()


+-------+--------+----+-------+---+
|   dept|employee|year|revenue| rn|
+-------+--------+----+-------+---+
|Finance|   Karan|2023|    200|  1|
|     HR|    Sita|2023|     80|  1|
|     IT|  Somesh|2023|    120|  1|
+-------+--------+----+-------+---+



Cumulative sum revenue

In [17]:
wc = Window.orderBy("year").rowsBetween(Window.unboundedPreceding,Window.currentRow)
df.withColumn("running_total", sum("revenue").over(wc)).show()


25/11/23 12:39:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+-------------+
|   dept|employee|year|revenue|running_total|
+-------+--------+----+-------+-------------+
|     IT|  Somesh|2023|    120|          120|
|     HR|    Sita|2023|     80|          200|
|Finance|   Karan|2023|    200|          400|
|     IT|  Ramesh|2024|    150|          550|
|     IT|   Kiran|2024|    180|          730|
|     HR|   Anita|2024|     90|          820|
|     HR|   Rekha|2024|    110|          930|
|Finance|    John|2024|    220|         1150|
|Finance|    Amit|2024|    250|         1400|
+-------+--------+----+-------+-------------+



25/11/23 12:39:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [16]:
wc = Window.orderBy("year")
df.withColumn("running_total", sum("revenue").over(wc)).show()


25/11/23 12:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+--------+----+-------+-------------+
|   dept|employee|year|revenue|running_total|
+-------+--------+----+-------+-------------+
|     IT|  Somesh|2023|    120|          400|
|     HR|    Sita|2023|     80|          400|
|Finance|   Karan|2023|    200|          400|
|     IT|  Ramesh|2024|    150|         1400|
|     IT|   Kiran|2024|    180|         1400|
|     HR|   Anita|2024|     90|         1400|
|     HR|   Rekha|2024|    110|         1400|
|Finance|    John|2024|    220|         1400|
|Finance|    Amit|2024|    250|         1400|
+-------+--------+----+-------+-------------+



25/11/23 12:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 12:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
